
# RNN vs LSTM vs GRU

## Instructions

In this lab, you will build **three text classification models** from scratch:
- RNN
- LSTM
- GRU

---

### Objectives
By the end of this lab, you should be able to:

- Preprocess text data
- Build a vocabulary
- Encode and pad sequences
- Implement RNN, LSTM, and GRU in PyTorch
- Train and evaluate models 
- Compare architectures


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from datasets import load_dataset

from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# data loading

dataset = load_dataset("ag_news")

# Reduce dataset size for faster training during class
train_data = list(dataset["train"])[:5000]
test_data = list(dataset["test"])[:1000]

print("Train size:", len(train_data))
print("Test size:", len(test_data))

README.md: 0.00B [00:00, ?B/s]

c:\Users\diogo\miniconda3\envs\Deep-Learning-Master\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diogo\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Train size: 5000
Test size: 1000



# Part 1 – Text Preprocessing

You must:

1. Write a `tokenize(text)` function.
2. Build a vocabulary using the training set only.
3. Keep only the top 10,000 most frequent words.
4. Add special tokens:
   - `<pad>`
   - `<unk>`
5. Explain in a markdown cell:
   - Why do we not build the vocabulary using the test set?


In [5]:
# Tokenization
def tokenize(text):
    return text.split()

# Build Vocabulary
def build_vocab(texts, max_words = 10000):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    most_common = Counter(counter).most_common(max_words - 2) # Reserve 2 for <PAD> (Padding) and <UK> (Unknown)

    vocab = {'<pad>': 0, '<unk>': 1}
    for i, (word, _) in enumerate(most_common, start=2):
        vocab[word] = i

    return vocab

In [8]:
vocab = build_vocab([item['text'] for item in train_data])
vocab

{'<pad>': 0,
 '<unk>': 1,
 'the': 2,
 'to': 3,
 'a': 4,
 'of': 5,
 'in': 6,
 'and': 7,
 'on': 8,
 '-': 9,
 'for': 10,
 'that': 11,
 'as': 12,
 'with': 13,
 '(Reuters)': 14,
 'at': 15,
 'The': 16,
 '#39;s': 17,
 'its': 18,
 'by': 19,
 'is': 20,
 'from': 21,
 'has': 22,
 'said': 23,
 'it': 24,
 'an': 25,
 'his': 26,
 '--': 27,
 '(AP)': 28,
 'was': 29,
 '...': 30,
 'after': 31,
 'will': 32,
 'U.S.': 33,
 'are': 34,
 'US': 35,
 'be': 36,
 'AP': 37,
 'have': 38,
 'their': 39,
 'Google': 40,
 'more': 41,
 'first': 42,
 'new': 43,
 'A': 44,
 'over': 45,
 'Tuesday': 46,
 'but': 47,
 'this': 48,
 'Inc.': 49,
 'up': 50,
 'Olympic': 51,
 'company': 52,
 'two': 53,
 'Wednesday': 54,
 'into': 55,
 'New': 56,
 'than': 57,
 'about': 58,
 'not': 59,
 'out': 60,
 'Reuters': 61,
 'NEW': 62,
 'he': 63,
 'they': 64,
 'prices': 65,
 'had': 66,
 'one': 67,
 'were': 68,
 'oil': 69,
 'United': 70,
 'Thursday': 71,
 'last': 72,
 'who': 73,
 'Iraq': 74,
 'could': 75,
 'YORK': 76,
 'been': 77,
 'against': 78,
 '

The test set is used only for testing our resolution, and not to use it for explore it. Building the vocabulary using the test set is considering *cheating* since, we are using information that was not supposed to know to build something that we need to know.


# Part 2 – Encoding and Padding

You must:

1. Create an `encode(text)` function.
2. Convert tokens into vocabulary indices.
3. Pad or truncate sequences to a fixed length (e.g., 25).
4. Create a custom `collate()` function.
5. Create train, validation, and test DataLoaders.

Explain:
- Why is padding necessary?
- Why should validation and test loaders not shuffle?


In [ ]:
def encode(text, vocab, max_lens = 25):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab['<unk>']) for token in tokens]

    return ids[: max_lens]

def collate


# Part 3 – Model Implementation

Create a class called `Model` that:

- Uses an Embedding layer
- Supports:
  - RNN
  - LSTM
  - GRU
- Uses multiple layers
- Applies dropout
- Outputs class logits

Your model must accept:
- model_type
- vocab_size
- embed_dim
- hidden_dim
- num_layers

Explain:
- The internal difference between RNN, LSTM, and GRU.



# Part 4 – Training Loop

Implement:

- A full training loop
- Validation loop
- Accuracy computation
- Loss tracking per epoch

Train for 10-50 epochs.

Store:
- train_loss
- val_loss
- train_accuracy
- val_accuracy

Explain:
- Why do we use `model.train()` and `model.eval()`?



# Part 5 – Model Comparison

Train:
- RNN
- LSTM
- GRU

Track validation accuracy and determine:
- Which performs best?
- Why?

Plot:
- Loss curves
- Accuracy curves

Explain signs of overfitting.



# Part 6 – Final Evaluation

Using the best model:

1. Evaluate on the test set.
2. Compute test accuracy.
3. Plot a confusion matrix.

Explain:
- Which classes are most confused?
- Why might that happen?
